# Проверка клинических гипотез

## Цель исследования

Цель ноутбука — проверить, наблюдаются ли в предоставленном датасете клинические закономерности, сформулированные на основании медицинской логики и экспертных рекомендаций.

Для каждой гипотезы проверяется:
1. Соответствует ли направление изменения лабораторных показателей ожидаемому.
2. Различаются ли распределения показателей между группами.
3. Достаточно ли доступных наблюдений.
4. Насколько выражены обнаруженные различия.

На этом этапе гипотезы не изменяют предсказания ML-модели. После проверки подтверждённые закономерности могут рассматриваться как engineered features, consistency rules или основа для объяснения предсказаний. После получения финальной модели результаты будут сопоставлены с ML и SHAP.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.stats import mannwhitneyu

In [2]:
DATA_PATH = Path("../data/raw/deficiency_anemia.csv")

df = pd.read_csv(DATA_PATH)

print("Размер датасета:", df.shape)
display(df.head())

Размер датасета: (840, 48)


,patient_id,age_years,sex,hemoglobin,RBC,hematocrit,MCV,MCH,MCHC,RDW,...,anemia,iron_deficiency,B12_deficiency,folate_deficiency,B6_deficiency,copper_deficiency,inflammation_anemia,mixed_deficiency,anemia_class,deficiency_cause
0,P000001,64,M,139.6,4.99,41.8,83.3,28.0,332.5,13.3,...,0,0,1,0,0,0,0,0,B12_deficiency_no_anemia,B12_deficiency
1,P000002,52,F,122.3,3.87,35.5,90.0,31.6,343.7,19.8,...,0,0,0,1,0,0,0,0,folate_deficiency_no_anemia,folate_deficiency
2,P000003,43,F,144.9,5.20,44.6,86.1,27.9,329.2,14.7,...,0,1,0,0,0,0,0,0,latent_deficiency,iron_deficiency
3,P000004,26,F,121.7,4.46,39.0,86.9,27.3,316.2,16.7,...,0,0,1,0,0,0,0,0,B12_deficiency_no_anemia,B12_deficiency
4,P000005,30,F,111.5,2.98,32.5,109.1,37.5,341.6,19.7,...,1,0,1,0,0,0,0,0,B12_deficiency_anemia,B12_deficiency


In [3]:
print("Количество колонок:", len(df.columns))
for col in df.columns:
    print(col)

Количество колонок: 48
patient_id
age_years
sex
hemoglobin
RBC
hematocrit
MCV
MCH
MCHC
RDW
platelets
WBC
reticulocytes
ferritin
serum_iron
transferrin
TIBC
UIBC
TSAT
sTfR
Ret_He
vitamin_B12
active_B12
MMA
homocysteine
folate
vitamin_B6
copper
ceruloplasmin
CRP
ESR
creatinine
eGFR
TSH
albumin
LDH
indirect_bilirubin
haptoglobin
anemia
iron_deficiency
B12_deficiency
folate_deficiency
B6_deficiency
copper_deficiency
inflammation_anemia
mixed_deficiency
anemia_class
deficiency_cause


## Методика проверки

Для большинства гипотез сравниваются две независимые группы: `target = 0` и `target = 1`.

Для каждого показателя рассчитываются количество наблюдений, медианы, ожидаемое и фактическое направление, Mann–Whitney U test и размер эффекта.

`p < 0.05` рассматривается как статистическое свидетельство различия распределений, но не как самостоятельное доказательство клинической значимости.

Итог оценивается по совокупности: **направление + статистическая значимость + размер эффекта + достаточность данных**.

In [4]:
def check_direction(median_0, median_1, expected):
    """Проверяет соответствие наблюдаемого изменения ожидаемому направлению."""
    if pd.isna(median_0) or pd.isna(median_1):
        return "недостаточно данных"
    if expected == "up":
        return "совпадает" if median_1 > median_0 else "не совпадает"
    if expected == "down":
        return "совпадает" if median_1 < median_0 else "не совпадает"
    return "неизвестное направление"


def rank_biserial_effect(U, n0, n1):
    """Рассчитывает rank-biserial correlation на основе Mann-Whitney U."""
    if n0 == 0 or n1 == 0:
        return np.nan
    return 2 * U / (n0 * n1) - 1


def effect_strength(effect):
    """Описательная классификация величины эффекта."""
    if pd.isna(effect):
        return "нет данных"
    value = abs(effect)
    if value < 0.1:
        return "очень слабый"
    if value < 0.3:
        return "слабый"
    if value < 0.5:
        return "средний"
    return "сильный"

In [5]:
def evaluate_binary_hypothesis(df, target, expected_directions):
    """Проверяет клиническую гипотезу для бинарного target."""
    results = []

    for feature, expected in expected_directions.items():
        group_0 = df.loc[df[target] == 0, feature].dropna()
        group_1 = df.loc[df[target] == 1, feature].dropna()

        median_0 = group_0.median()
        median_1 = group_1.median()
        direction = check_direction(median_0, median_1, expected)

        if len(group_0) > 0 and len(group_1) > 0:
            U, p = mannwhitneyu(group_0, group_1, alternative="two-sided")
            effect = rank_biserial_effect(U, len(group_0), len(group_1))
        else:
            U, p, effect = np.nan, np.nan, np.nan

        results.append({
            "feature": feature,
            "expected": expected,
            "n_0": len(group_0),
            "n_1": len(group_1),
            "median_0": median_0,
            "median_1": median_1,
            "direction": direction,
            "U": U,
            "p_value": p,
            "significant_0.05": p < 0.05 if not pd.isna(p) else False,
            "effect_size": effect,
            "effect_strength": effect_strength(effect),
        })

    return pd.DataFrame(results)


def summarize_hypothesis(results):
    """Выводит краткую сводку результатов."""
    total = len(results)
    matched = (results["direction"] == "совпадает").sum()
    significant = results["significant_0.05"].sum()
    both = ((results["direction"] == "совпадает") & results["significant_0.05"]).sum()

    print(f"Всего показателей: {total}")
    print(f"Направление совпало: {matched}/{total}")
    print(f"p < 0.05: {significant}/{total}")
    print(f"Направление совпало и p < 0.05: {both}/{total}")

# H1. Лабораторный профиль железодефицита

Ожидается: ferritin ↓, serum iron ↓, TSAT ↓, TIBC ↑, transferrin ↑, MCV ↓, MCH ↓, RDW ↑.

Проверяем, действительно ли при `iron_deficiency = 1` показатели изменяются в ожидаемом направлении относительно `iron_deficiency = 0`.

In [6]:
expected_h1 = {
    "ferritin": "down",
    "serum_iron": "down",
    "TSAT": "down",
    "TIBC": "up",
    "transferrin": "up",
    "MCV": "down",
    "MCH": "down",
    "RDW": "up",
}

h1_results = evaluate_binary_hypothesis(df, "iron_deficiency", expected_h1)
display(h1_results)
summarize_hypothesis(h1_results)

,feature,expected,n_0,n_1,median_0,median_1,direction,U,p_value,significant_0.05,effect_size,effect_strength
0,ferritin,down,331,263,94.80,13.10,совпадает,85623.5,2.768666e-91,True,0.967158,сильный
1,serum_iron,down,297,243,17.60,8.30,совпадает,62737.0,2.116981e-49,True,0.738565,сильный
2,TSAT,down,297,243,27.30,10.30,совпадает,68709.0,4.114122e-73,True,0.904061,сильный
3,TIBC,up,297,243,61.00,80.10,совпадает,6487.0,1.654913e-60,True,-0.820233,сильный
4,transferrin,up,297,243,2.53,3.21,совпадает,8048.5,1.762419e-54,True,-0.776960,сильный
5,MCV,down,550,290,91.05,84.00,совпадает,124160.5,2.898675e-40,True,0.556871,сильный
6,MCH,down,550,290,30.10,26.40,совпадает,130349.5,9.552855e-52,True,0.634476,сильный
7,RDW,up,541,284,14.60,17.65,совпадает,33941.5,1.045629e-39,True,-0.558180,сильный


Всего показателей: 8
Направление совпало: 8/8
p < 0.05: 8/8
Направление совпало и p < 0.05: 8/8


### Интерпретация H1

Оцениваем совпадение направлений, статистическую значимость, величину эффекта и количество наблюдений. Даже подтверждение H1 пока не означает, что правило нужно добавлять в ML: это будет проверено отдельно после получения финальной модели.

# H2. Железодефицит vs анемия воспаления

При железодефиците ожидаются ferritin ↓, TIBC ↑, transferrin ↑, sTfR ↑.

При воспалительном состоянии ожидаются более высокий ferritin, CRP ↑, ESR ↑, более низкие TIBC, transferrin и sTfR.

Serum iron и TSAT могут снижаться в обоих состояниях, поэтому рассматриваются совместно с другими маркерами.

In [7]:
h2_df = df[
    ((df["iron_deficiency"] == 1) & (df["inflammation_anemia"] == 0))
    |
    ((df["iron_deficiency"] == 0) & (df["inflammation_anemia"] == 1))
].copy()

h2_df["h2_group"] = np.where(
    h2_df["iron_deficiency"] == 1,
    "iron_deficiency",
    "inflammation",
)

display(h2_df["h2_group"].value_counts())

h2_group
iron_deficiency    290
inflammation        65
Name: count, dtype: int64

In [8]:
h2_features = [
    "ferritin", "serum_iron", "TSAT", "TIBC",
    "transferrin", "sTfR", "CRP", "ESR",
]

h2_medians = h2_df.groupby("h2_group")[h2_features].median().T
display(h2_medians)

h2_group,inflammation,iron_deficiency
ferritin,200.60,13.10
serum_iron,6.70,8.30
TSAT,14.50,10.30
TIBC,49.50,80.10
transferrin,2.18,3.21
sTfR,1.60,3.90
CRP,41.40,1.70
ESR,40.00,9.00


In [9]:
expected_h2 = {
    "ferritin": "higher_in_inflammation",
    "serum_iron": "similar_or_variable",
    "TSAT": "similar_or_variable",
    "TIBC": "lower_in_inflammation",
    "transferrin": "lower_in_inflammation",
    "sTfR": "lower_in_inflammation",
    "CRP": "higher_in_inflammation",
    "ESR": "higher_in_inflammation",
}

def check_h2_direction(iron_value, inflammation_value, expected):
    if pd.isna(iron_value) or pd.isna(inflammation_value):
        return "недостаточно данных"
    if expected == "higher_in_inflammation":
        return "совпадает" if inflammation_value > iron_value else "не совпадает"
    if expected == "lower_in_inflammation":
        return "совпадает" if inflammation_value < iron_value else "не совпадает"
    if expected == "similar_or_variable":
        return "направление не фиксируем"
    return "неизвестно"

In [10]:
h2_results = []

for feature, expected in expected_h2.items():
    iron = h2_df.loc[h2_df["h2_group"] == "iron_deficiency", feature].dropna()
    inflammation = h2_df.loc[h2_df["h2_group"] == "inflammation", feature].dropna()

    median_iron = iron.median()
    median_inflammation = inflammation.median()
    direction = check_h2_direction(median_iron, median_inflammation, expected)

    if len(iron) > 0 and len(inflammation) > 0:
        U, p = mannwhitneyu(iron, inflammation, alternative="two-sided")
        effect = rank_biserial_effect(U, len(iron), len(inflammation))
    else:
        U, p, effect = np.nan, np.nan, np.nan

    h2_results.append({
        "feature": feature,
        "expected": expected,
        "n_iron": len(iron),
        "n_inflammation": len(inflammation),
        "median_iron": median_iron,
        "median_inflammation": median_inflammation,
        "direction": direction,
        "p_value": p,
        "significant_0.05": p < 0.05 if not pd.isna(p) else False,
        "effect_size": effect,
        "effect_strength": effect_strength(effect),
    })

h2_results = pd.DataFrame(h2_results)
display(h2_results)

,feature,expected,n_iron,n_inflammation,median_iron,median_inflammation,direction,p_value,significant_0.05,effect_size,effect_strength
0,ferritin,higher_in_inflammation,263,54,13.10,200.60,совпадает,5.555870e-31,True,-1.000000,сильный
1,serum_iron,similar_or_variable,243,55,8.30,6.70,направление не фиксируем,6.278691e-04,True,0.295324,слабый
2,TSAT,similar_or_variable,243,55,10.30,14.50,направление не фиксируем,2.401934e-08,True,-0.481930,средний
3,TIBC,lower_in_inflammation,243,55,80.10,49.50,совпадает,1.021008e-30,True,0.995062,сильный
4,transferrin,lower_in_inflammation,243,55,3.21,2.18,совпадает,1.176964e-28,True,0.959072,сильный
5,sTfR,lower_in_inflammation,91,20,3.90,1.60,совпадает,2.982874e-12,True,1.000000,сильный
6,CRP,higher_in_inflammation,158,57,1.70,41.40,совпадает,1.406437e-28,True,-0.991450,сильный
7,ESR,higher_in_inflammation,161,59,9.00,40.00,совпадает,1.703006e-25,True,-0.917781,сильный


### Интерпретация H2

Оцениваем, различаются ли профили железодефицита и анемии воспаления в ожидаемом направлении. Особый интерес представляют ferritin, TIBC/transferrin, sTfR и CRP/ESR. Serum iron и TSAT интерпретируются только в составе комплексного профиля.

# H3. Латентный железодефицит

Ожидаемый профиль: ferritin ↓, TSAT ↓, sTfR ↑, Ret-He ↓.

Для корректной проверки нужна независимая метка латентного железодефицита. Если её нет, мы не создаём target из тех же показателей, которые хотим проверять, чтобы избежать круговой проверки.

In [11]:
has_latent_target = "latent_deficiency" in df.columns
print("Есть target latent_deficiency:", has_latent_target)

if has_latent_target:
    expected_h3 = {
        "ferritin": "down",
        "TSAT": "down",
        "sTfR": "up",
        "Ret_He": "down",
    }
    h3_results = evaluate_binary_hypothesis(df, "latent_deficiency", expected_h3)
    display(h3_results)
    summarize_hypothesis(h3_results)
else:
    print("H3 не проверяется: независимый target отсутствует.")

Есть target latent_deficiency: False
H3 не проверяется: независимый target отсутствует.


# H4. Функциональный профиль B12-дефицита

Ожидается: vitamin B12 ↓, active B12 ↓, MMA ↑, homocysteine ↑.

In [12]:
expected_h4 = {
    "vitamin_B12": "down",
    "active_B12": "down",
    "MMA": "up",
    "homocysteine": "up",
}

h4_results = evaluate_binary_hypothesis(df, "B12_deficiency", expected_h4)
display(h4_results)
summarize_hypothesis(h4_results)

,feature,expected,n_0,n_1,median_0,median_1,direction,U,p_value,significant_0.05,effect_size,effect_strength
0,vitamin_B12,down,347,166,424.0000,166.000,совпадает,56017.0,2.963564e-67,True,0.944967,сильный
1,active_B12,down,165,87,82.3000,25.000,совпадает,14328.0,1.275421e-38,True,0.996238,сильный
2,MMA,up,174,98,0.1875,0.629,совпадает,181.0,6.239305e-41,True,-0.978771,сильный
3,homocysteine,up,199,99,12.7000,28.000,совпадает,2099.5,1.904224e-28,True,-0.786864,сильный


Всего показателей: 4
Направление совпало: 4/4
p < 0.05: 4/4
Направление совпало и p < 0.05: 4/4


# H5. B12-дефицит vs фолатный дефицит

Оба состояния могут сопровождаться повышением homocysteine. Для B12-дефицита дополнительно ожидается MMA ↑, тогда как выраженное повышение MMA не рассматривается как типичный признак изолированного фолатного дефицита.

In [ ]:
h5_df = df[
    ((df["B12_deficiency"] == 1) & (df["folate_deficiency"] == 0))
    |
    ((df["B12_deficiency"] == 0) & (df["folate_deficiency"] == 1))
].copy()

h5_df["h5_group"] = np.where(h5_df["B12_deficiency"] == 1, "B12", "folate")

display(h5_df["h5_group"].value_counts())

h5_features = [
    "vitamin_B12", "active_B12", "folate",
    "MMA", "homocysteine", "MCV", "RDW",
]

display(h5_df.groupby("h5_group")[h5_features].agg(["count", "median"]))

In [ ]:
b12_mma = h5_df.loc[h5_df["h5_group"] == "B12", "MMA"].dropna()
folate_mma = h5_df.loc[h5_df["h5_group"] == "folate", "MMA"].dropna()

if len(b12_mma) > 0 and len(folate_mma) > 0:
    U, p = mannwhitneyu(b12_mma, folate_mma, alternative="two-sided")
    effect = rank_biserial_effect(U, len(b12_mma), len(folate_mma))

    print("Median MMA — B12:", b12_mma.median())
    print("Median MMA — folate:", folate_mma.median())
    print("p-value:", p)
    print("effect size:", effect)
    print("effect strength:", effect_strength(effect))
else:
    print("Недостаточно данных MMA для сравнения.")

# H6. Смешанный дефицит

При сочетании дефицитов с противоположным влиянием на размер эритроцитов MCV может оставаться относительно нормальным, а RDW повышаться из-за большей неоднородности эритроцитов.

In [ ]:
expected_h6 = {"RDW": "up"}

h6_results = evaluate_binary_hypothesis(df, "mixed_deficiency", expected_h6)
display(h6_results)
summarize_hypothesis(h6_results)

display(
    df.groupby("mixed_deficiency")["MCV"]
    .agg(["count", "median", "mean", "std"])
)

# H7. Дефицит меди

Ожидаются copper ↓ и ceruloplasmin ↓. Дополнительно исследуются WBC и эритроцитарные индексы. Учитываем возможное большое число пропусков специализированных анализов.

In [ ]:
expected_h7 = {
    "copper": "down",
    "ceruloplasmin": "down",
}

h7_results = evaluate_binary_hypothesis(df, "copper_deficiency", expected_h7)
display(h7_results)
summarize_hypothesis(h7_results)

display(
    df.groupby("copper_deficiency")[["WBC", "MCV", "MCH", "hemoglobin"]]
    .agg(["count", "median"])
)

# H8. Дефицит витамина B6

Основное ожидаемое изменение — vitamin B6 ↓. Дополнительно исследуются MCV, MCH и показатели обмена железа как исследовательские, а не специфические диагностические критерии.

In [ ]:
expected_h8 = {"vitamin_B6": "down"}

h8_results = evaluate_binary_hypothesis(df, "B6_deficiency", expected_h8)
display(h8_results)
summarize_hypothesis(h8_results)

display(
    df.groupby("B6_deficiency")[
        ["MCV", "MCH", "ferritin", "serum_iron", "TSAT"]
    ]
    .agg(["count", "median"])
)

# H9. Гемолитический лабораторный профиль

Исследуем сочетание LDH ↑, indirect bilirubin ↑, haptoglobin ↓, reticulocytes ↑ среди случаев анемии другого происхождения.

Это не постановка диагноза гемолиза, а проверка наличия соответствующего лабораторного профиля. Перед формированием группы смотрим фактические значения `anemia_class`.

In [ ]:
display(df["anemia_class"].value_counts(dropna=False))

# H10. Почечный и тиреоидный профиль при анемии другого происхождения

Исследуем почечный профиль (creatinine ↑, eGFR ↓) и тиреоидный профиль (TSH ↑) среди случаев анемии другого происхождения.

Цель — не диагностировать заболевание почек или гипотиреоз, а проверить наличие соответствующих лабораторных закономерностей. Формирование группы выполняется после проверки фактических значений `anemia_class`.

# Сводка клинических гипотез

Для каждой гипотезы фиксируются соответствие направлению, статистическая значимость, величина эффекта, ограничения по данным и решение о дальнейшем использовании.

Возможные решения:
- `candidate_feature`
- `rule`
- `explanation`
- `research_only`
- `insufficient_data`

In [ ]:
hypothesis_summary = pd.DataFrame([
    ["H1", "Iron deficiency profile", "checked", "candidate_feature / explanation"],
    ["H2", "Iron deficiency vs inflammation", "checked", "candidate_feature / rule / explanation"],
    ["H3", "Latent iron deficiency", "depends_on_target", "check target availability"],
    ["H4", "B12 functional profile", "checked", "candidate_feature / explanation"],
    ["H5", "B12 vs folate", "checked", "candidate_feature / explanation"],
    ["H6", "Mixed deficiency", "checked", "candidate_feature / explanation"],
    ["H7", "Copper deficiency", "checked", "evaluate missingness"],
    ["H8", "B6 deficiency", "checked", "evaluate missingness"],
    ["H9", "Hemolysis profile", "pending_anemia_class", "research / rule"],
    ["H10", "Renal / thyroid profile", "pending_anemia_class", "research / rule"],
], columns=["hypothesis", "name", "status", "next_step"])

display(hypothesis_summary)

# Следующий этап

После получения финальной ML-модели:

1. Зафиксировать baseline Random Forest.
2. Формализовать подтверждённые гипотезы как candidate clinical features.
3. Сравнить baseline и baseline + clinical features на одинаковых CV folds.
4. Проверить Macro F1, Balanced Accuracy и per-class Recall.
5. Выполнить ablation study.
6. Рассчитать SHAP.
7. Сопоставить поведение модели с клиническими закономерностями.
8. Определить, что пойдёт в engineered features, consistency rules, explanations или останется только исследовательским результатом.

Только после этого практически полезные правила переносятся из исследовательского ноутбука в production-код.